In [1]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error
import joblib
from lightgbm import LGBMRegressor

# =========================
# CONFIG
# =========================
BASE_DIR = Path(r"C:/Plegma_Programming")
DATA_PATH = BASE_DIR / "PLEGMA_final_hourly_dataset.csv"
OUT_DIR = BASE_DIR / "models" / "final_lgbm"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TIME_COL = "timestamp"
HOME_COL = "home_id"
TARGET_COL = "consumption_Wh"

TEST_DAYS = 30
RANDOM_STATE = 42
SPLIT_MODE = "global"   # keep same style as RF

USE_LOG_TARGET = True
CLIP_TARGET_TRAIN = True
CLIP_Q_LOW = 0.005
CLIP_Q_HIGH = 0.995

CATEGORICAL = [
    "building_type",
    "build_era",
    "season",
    "income_band",
    "heating_type",
    "water_heater_type",
    "homeowner_status",
    "years_in_house",
    "occupation",
]

NUMERIC_CORE = [
    "hour",
    "day_of_week",
    "is_weekend",
    "is_holiday",
    "month",
    "internal_temperature",
    "external_temperature",
    "internal_humidity",
    "external_humidity",
    "num_rooms",
    "residents",
    "num_adults",
    "num_children",
    "num_elderly",
    "has_ac",
    "has_fridge_freezer",
    "has_dryer",
    "has_washing_machine",
    "has_dishwasher",
    "has_microwave",
    "has_electric_oven",
    "has_electric_hob",
    "solar_panels",
]

LAG_FEATURES = [
    "lag_1h",
    "lag_24h",
    "lag_168h",
    "roll_mean_24h",
    "roll_mean_168h",
]

LGBM_PARAMS = dict(
    n_estimators=400,
    learning_rate=0.05,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

# =========================
# METRICS
# =========================
def safe_mape(y_true, y_pred, eps=1e-9):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    den = np.where(np.abs(y_true) < eps, np.nan, y_true)
    return float(np.nanmean(np.abs((y_true - y_pred) / den)) * 100)


def compute_metrics(y_true, y_pred):
    return {
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "MAPE_%": safe_mape(y_true, y_pred),
    }


# =========================
# HELPERS
# =========================
def save_json(obj, path: Path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def clean_types(df: pd.DataFrame) -> pd.DataFrame:
    numeric_cols = NUMERIC_CORE + [TARGET_COL]
    for c in numeric_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    for c in CATEGORICAL:
        if c in df.columns:
            df[c] = df[c].astype("string").fillna("Unknown")

    df = df.dropna(subset=[TARGET_COL]).copy()
    return df


def make_onehot(df: pd.DataFrame, feature_cols: list, categorical_cols: list) -> pd.DataFrame:
    cat_cols_present = [c for c in categorical_cols if c in feature_cols]
    return pd.get_dummies(df[feature_cols], columns=cat_cols_present, drop_first=False)


def align_test_to_train(X_train: pd.DataFrame, X_test: pd.DataFrame) -> pd.DataFrame:
    return X_test.reindex(columns=X_train.columns, fill_value=0)


def time_split_global(df: pd.DataFrame, test_days: int = 30):
    cutoff = df[TIME_COL].max() - pd.Timedelta(days=test_days)
    train_df = df[df[TIME_COL] <= cutoff].copy()
    test_df = df[df[TIME_COL] > cutoff].copy()
    return train_df, test_df, cutoff


def train_only_impute(train_df: pd.DataFrame, test_df: pd.DataFrame, cols: list):
    medians = {}
    for c in cols:
        med = float(train_df[c].median()) if train_df[c].notna().any() else 0.0
        medians[c] = med
        train_df[c] = train_df[c].fillna(med)
        test_df[c] = test_df[c].fillna(med)
    return train_df, test_df, medians


def maybe_clip_train_target(y_train):
    if not CLIP_TARGET_TRAIN:
        return y_train, None
    lo = np.quantile(y_train, CLIP_Q_LOW)
    hi = np.quantile(y_train, CLIP_Q_HIGH)
    y_train_clipped = np.clip(y_train, lo, hi)
    return y_train_clipped, (float(lo), float(hi))


def y_transform_train(y):
    if USE_LOG_TARGET:
        return np.log1p(np.maximum(y, 0.0))
    return y


def y_inverse_pred(y_hat):
    if USE_LOG_TARGET:
        return np.expm1(y_hat)
    return y_hat


# =========================
# LOAD DATA
# =========================
df = pd.read_csv(DATA_PATH, parse_dates=[TIME_COL], low_memory=False)
df = df.sort_values([HOME_COL, TIME_COL]).reset_index(drop=True)
df = clean_types(df)

df_global_time = df.sort_values(TIME_COL).reset_index(drop=True)

results_summary = []

# =========================
# MODE 1: COLD-START
# =========================
FEATURES_COLD = NUMERIC_CORE + CATEGORICAL

train_cold, test_cold, cutoff_cold = time_split_global(df_global_time, TEST_DAYS)

num_cols_cold = [c for c in NUMERIC_CORE if c not in CATEGORICAL]
train_cold, test_cold, medians_cold = train_only_impute(train_cold, test_cold, num_cols_cold)

for c in num_cols_cold:
    train_cold[c] = train_cold[c].astype("float32")
    test_cold[c] = test_cold[c].astype("float32")

train_cold[TARGET_COL] = train_cold[TARGET_COL].astype("float32")
test_cold[TARGET_COL] = test_cold[TARGET_COL].astype("float32")

X_train_cold = make_onehot(train_cold, FEATURES_COLD, CATEGORICAL).astype("float32")
X_test_cold = make_onehot(test_cold, FEATURES_COLD, CATEGORICAL).astype("float32")
X_test_cold = align_test_to_train(X_train_cold, X_test_cold).astype("float32")

y_train_cold = train_cold[TARGET_COL].to_numpy(dtype="float32")
y_test_cold = test_cold[TARGET_COL].to_numpy(dtype="float32")

y_train_cold_used, clip_bounds_cold = maybe_clip_train_target(y_train_cold)
y_train_cold_tr = y_transform_train(y_train_cold_used)

lgbm_cold = LGBMRegressor(**LGBM_PARAMS)
print(f"\nTraining LGBM (cold-start) | SPLIT_MODE={SPLIT_MODE} | LOG={USE_LOG_TARGET} | CLIP={CLIP_TARGET_TRAIN} ...")
lgbm_cold.fit(X_train_cold, y_train_cold_tr)

pred_cold_tr = lgbm_cold.predict(X_test_cold).astype("float32")
pred_cold = y_inverse_pred(pred_cold_tr).astype("float32")
pred_cold = np.maximum(pred_cold, 0.0)

metrics_cold = compute_metrics(y_test_cold, pred_cold)
print("LGBM cold-start metrics:", metrics_cold)

joblib.dump(lgbm_cold, OUT_DIR / "lgbm_coldstart.joblib")
save_json(list(X_train_cold.columns), OUT_DIR / "lgbm_coldstart_feature_columns.json")
save_json(medians_cold, OUT_DIR / "lgbm_coldstart_train_medians.json")
save_json(
    {
        "split_mode": SPLIT_MODE,
        "test_days": TEST_DAYS,
        "cutoff_global": str(cutoff_cold) if cutoff_cold is not None else None,
        "use_log_target": USE_LOG_TARGET,
        "clip_target_train": CLIP_TARGET_TRAIN,
        "clip_bounds": clip_bounds_cold,
        "lgbm_params": LGBM_PARAMS,
        "features_cold": FEATURES_COLD,
        "categorical": CATEGORICAL,
    },
    OUT_DIR / "lgbm_coldstart_train_extras.json",
)

preds_cold_df = test_cold[[HOME_COL, TIME_COL, TARGET_COL]].copy()
preds_cold_df["prediction"] = pred_cold
preds_cold_df.to_csv(OUT_DIR / "lgbm_coldstart_predictions.csv", index=False)

results_summary.append({"mode": "coldstart", **metrics_cold})

# =========================
# MODE 2: WITH-HISTORY
# =========================
df_lags = df.sort_values([HOME_COL, TIME_COL]).reset_index(drop=True)
g = df_lags.groupby(HOME_COL, sort=False)[TARGET_COL]

df_lags["lag_1h"] = g.shift(1)
df_lags["lag_24h"] = g.shift(24)
df_lags["lag_168h"] = g.shift(168)
df_lags["roll_mean_24h"] = g.shift(1).rolling(24).mean()
df_lags["roll_mean_168h"] = g.shift(1).rolling(168).mean()

df_lags = df_lags.dropna(subset=LAG_FEATURES).copy()

for c in LAG_FEATURES:
    df_lags[c] = pd.to_numeric(df_lags[c], errors="coerce")

df_lags_global = df_lags.sort_values(TIME_COL).reset_index(drop=True)

FEATURES_HIST = NUMERIC_CORE + LAG_FEATURES + CATEGORICAL

train_hist, test_hist, cutoff_hist = time_split_global(df_lags_global, TEST_DAYS)

num_cols_hist = [c for c in (NUMERIC_CORE + LAG_FEATURES) if c not in CATEGORICAL]
train_hist, test_hist, medians_hist = train_only_impute(train_hist, test_hist, num_cols_hist)

for c in num_cols_hist:
    train_hist[c] = train_hist[c].astype("float32")
    test_hist[c] = test_hist[c].astype("float32")

train_hist[TARGET_COL] = train_hist[TARGET_COL].astype("float32")
test_hist[TARGET_COL] = test_hist[TARGET_COL].astype("float32")

X_train_hist = make_onehot(train_hist, FEATURES_HIST, CATEGORICAL).astype("float32")
X_test_hist = make_onehot(test_hist, FEATURES_HIST, CATEGORICAL).astype("float32")
X_test_hist = align_test_to_train(X_train_hist, X_test_hist).astype("float32")

y_train_hist = train_hist[TARGET_COL].to_numpy(dtype="float32")
y_test_hist = test_hist[TARGET_COL].to_numpy(dtype="float32")

y_train_hist_used, clip_bounds_hist = maybe_clip_train_target(y_train_hist)
y_train_hist_tr = y_transform_train(y_train_hist_used)

lgbm_hist = LGBMRegressor(**LGBM_PARAMS)
print(f"\nTraining LGBM (with-history) | SPLIT_MODE={SPLIT_MODE} | LOG={USE_LOG_TARGET} | CLIP={CLIP_TARGET_TRAIN} ...")
lgbm_hist.fit(X_train_hist, y_train_hist_tr)

pred_hist_tr = lgbm_hist.predict(X_test_hist).astype("float32")
pred_hist = y_inverse_pred(pred_hist_tr).astype("float32")
pred_hist = np.maximum(pred_hist, 0.0)

metrics_hist = compute_metrics(y_test_hist, pred_hist)
print("LGBM with-history metrics:", metrics_hist)

joblib.dump(lgbm_hist, OUT_DIR / "lgbm_withhistory.joblib")
save_json(list(X_train_hist.columns), OUT_DIR / "lgbm_withhistory_feature_columns.json")
save_json(medians_hist, OUT_DIR / "lgbm_withhistory_train_medians.json")
save_json(
    {
        "split_mode": SPLIT_MODE,
        "test_days": TEST_DAYS,
        "cutoff_global": str(cutoff_hist) if cutoff_hist is not None else None,
        "use_log_target": USE_LOG_TARGET,
        "clip_target_train": CLIP_TARGET_TRAIN,
        "clip_bounds": clip_bounds_hist,
        "lgbm_params": LGBM_PARAMS,
        "features_hist": FEATURES_HIST,
        "lag_features": LAG_FEATURES,
        "categorical": CATEGORICAL,
    },
    OUT_DIR / "lgbm_withhistory_train_extras.json",
)

preds_hist_df = test_hist[[HOME_COL, TIME_COL, TARGET_COL]].copy()
preds_hist_df["prediction"] = pred_hist
preds_hist_df.to_csv(OUT_DIR / "lgbm_withhistory_predictions.csv", index=False)

results_summary.append({"mode": "withhistory", **metrics_hist})

# =========================
# SUMMARY / CONFIG
# =========================
config = {
    "data_path": str(DATA_PATH),
    "target": TARGET_COL,
    "time_col": TIME_COL,
    "home_col": HOME_COL,
    "test_days": TEST_DAYS,
    "split_mode": SPLIT_MODE,
    "use_log_target": USE_LOG_TARGET,
    "clip_target_train": CLIP_TARGET_TRAIN,
    "clip_q_low": CLIP_Q_LOW,
    "clip_q_high": CLIP_Q_HIGH,
    "lgbm_params": LGBM_PARAMS,
    "features_cold_numeric_core": NUMERIC_CORE,
    "features_hist_lags": LAG_FEATURES,
    "categorical": CATEGORICAL,
    "metrics_cold": metrics_cold,
    "metrics_withhistory": metrics_hist,
}
save_json(config, OUT_DIR / "training_config.json")

metrics_df = pd.DataFrame(results_summary)
metrics_df.to_csv(OUT_DIR / "lgbm_metrics_summary.csv", index=False)

print("\nSaved artifacts:")
print(" -", OUT_DIR / "lgbm_coldstart.joblib")
print(" -", OUT_DIR / "lgbm_coldstart_feature_columns.json")
print(" -", OUT_DIR / "lgbm_coldstart_train_medians.json")
print(" -", OUT_DIR / "lgbm_coldstart_train_extras.json")
print(" -", OUT_DIR / "lgbm_coldstart_predictions.csv")
print(" -", OUT_DIR / "lgbm_withhistory.joblib")
print(" -", OUT_DIR / "lgbm_withhistory_feature_columns.json")
print(" -", OUT_DIR / "lgbm_withhistory_train_medians.json")
print(" -", OUT_DIR / "lgbm_withhistory_train_extras.json")
print(" -", OUT_DIR / "lgbm_withhistory_predictions.csv")
print(" -", OUT_DIR / "lgbm_metrics_summary.csv")
print(" -", OUT_DIR / "training_config.json")


Training LGBM (cold-start) | SPLIT_MODE=global | LOG=True | CLIP=True ...
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005319 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1162
[LightGBM] [Info] Number of data points in the train set: 64635, number of used features: 52
[LightGBM] [Info] Start training from score 5.113769
LGBM cold-start metrics: {'MAE': 190.34481811523438, 'RMSE': 426.2956903664404, 'MAPE_%': 56.75560712251722}

Training LGBM (with-history) | SPLIT_MODE=global | LOG=True | CLIP=True ...
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005535 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memo